In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from google.genai import types
from google.adk.sessions import DatabaseSessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from storybook_agent.agent import root_agent

artifact_service = InMemoryArtifactService()

session_service = DatabaseSessionService(db_url="sqlite:///../session.db")

session = await session_service.create_session(
    app_name="storybook_agent",
    user_id="u_123",
    state={},
)

runner = Runner(
    agent=root_agent,
    session_service=session_service,
    app_name="storybook_agent",
    artifact_service=artifact_service,
)

message = types.Content(
    role="user",
    parts=[types.Part(text="테마: 용감한 토끼 베니의 모험")],
)

async for event in runner.run_async(
    user_id="u_123", session_id=session.id, new_message=message
):
    if event.is_final_response():
        print(event.content.parts[0].text)
    else:
        print(event.get_function_calls())
        print(event.get_function_responses())